In [ ]:
import pandas as pd

data = pd.read_csv("../data/Clean_Dataset.csv")

data = data.rename(columns={"Unnamed: 0": "id"})
data.drop_duplicates()

columns = []

for single_data in data :
    if data[single_data][0] == str:
        columns.append(single_data)

for col in columns:
    data[col] = data[col].str.strip()

stop_mapping = {
    "zero": 0,
    "one": 1,
    "two": 2
}

data["stops"] = data["stops"].map(stop_mapping)
data["days_left"] = data["days_left"].fillna(data["days_left"].median())
data["price"] = data["price"].astype(float)
data["days_left"] = data["days_left"].astype(float)
data["duration"] = data["duration"].astype(float)

data = data.replace("", pd.NA)

print(data)


            id   airline   flight source_city departure_time  stops  \
0            0  SpiceJet  SG-8709       Delhi        Evening    0.0   
1            1  SpiceJet  SG-8157       Delhi  Early_Morning    0.0   
2            2   AirAsia   I5-764       Delhi  Early_Morning    0.0   
3            3   Vistara   UK-995       Delhi        Morning    0.0   
4            4   Vistara   UK-963       Delhi        Morning    0.0   
...        ...       ...      ...         ...            ...    ...   
300148  300148   Vistara   UK-822     Chennai        Morning    1.0   
300149  300149   Vistara   UK-826     Chennai      Afternoon    1.0   
300150  300150   Vistara   UK-832     Chennai  Early_Morning    1.0   
300151  300151   Vistara   UK-828     Chennai  Early_Morning    1.0   
300152  300152   Vistara   UK-822     Chennai        Morning    1.0   

         arrival_time destination_city     class  duration  days_left    price  
0               Night           Mumbai   Economy      2.17        

In [ ]:
categorical = ["stops", "class", "departure_time", "arrival_time"]
ordinal = ["airline", "flight", "destination_city"]
quantitative = ["duration", "days_left", "price"]
cible = ["price"]

for category in categorical:
    freq = data[category].value_counts()
    rep = (data[category].value_counts() / data[category].count()) * 100
    dom = rep.index[0]
    max_pct = rep.iloc[0]
    min_pct = rep.iloc[-1]  

    print(f"{rep.index[0]} {max_pct}")
    print(f"{rep.index[-1]} {min_pct}")
    print(f"{rep.index[0]} is the most chosen")

stats = data.describe()

print(stats)

one 83.57837502873534
two_or_more 4.4264091979756985
one is the most chosen
Economy 68.85355135547537
Business 31.146448644524625
Economy is the most chosen
Morning 23.703244678547275
Late_Night 0.4351114265058154
Morning is the most chosen
Night 30.497113138965794
Late_Night 4.664621043267933
Night is the most chosen


In [3]:
    
import seaborn as sns
import matplotlib.pyplot as plt


sns.scatterplot(data['price'])
plt.show()


AttributeError: module 'seaborn' has no attribute 'scatter'

In [ ]:
price_per_hour = data["price_per_hour"] = data["price"] / data["duration"]
price_by_stop = data.groupby("stops")["price"].sum()
airline_revenue_by_flight = data.groupby("airline")["price"].mean()
price_by_days_left = data.groupby("days_left")["price"].sum()
print(price_per_hour)
print(airline_revenue_by_flight)
print(price_by_stop)
print(price_by_days_left)

0         2743.317972
1         2554.935622
2         2744.700461
3         2646.666667
4         2555.793991
             ...     
300148    6871.527778
300149    7399.712092
300150    5719.378163
300151    8158.500000
300152    8093.750000
Length: 300153, dtype: float64
airline
AirAsia       4091.072742
Air_India    23507.019112
GO_FIRST      5652.007595
Indigo        5324.216303
SpiceJet      6179.278881
Vistara      30396.536302
Name: price, dtype: float64
stops
one            5.745012e+09
two_or_more    1.875113e+08
zero           3.375713e+08
Name: price, dtype: float64
days_left
1.0      41607528.0
2.0     121630693.0
3.0     123090403.0
4.0     130635808.0
5.0     143857338.0
6.0     142676275.0
7.0     145930459.0
8.0     143574563.0
9.0     145739184.0
10.0    148884953.0
11.0    147531040.0
12.0    143609531.0
13.0    144082862.0
14.0    143982637.0
15.0    139179109.0
16.0    128598242.0
17.0    130860006.0
18.0    131957113.0
19.0    127521687.0
20.0    128089292.0
21.0   

In [ ]:

encoded_cat = pd.get_dummies(data, columns=categorical, drop_first=True, dtype=int)
print(encoded_cat)
target = data["price"]
exp = data[["airline", "flight", "duration", "days_left", "source_city", "destination_city", "days_left", "class", "stops", "arrival_time"]]

            id   airline   flight source_city destination_city  duration  \
0            0  SpiceJet  SG-8709       Delhi           Mumbai      2.17   
1            1  SpiceJet  SG-8157       Delhi           Mumbai      2.33   
2            2   AirAsia   I5-764       Delhi           Mumbai      2.17   
3            3   Vistara   UK-995       Delhi           Mumbai      2.25   
4            4   Vistara   UK-963       Delhi           Mumbai      2.33   
...        ...       ...      ...         ...              ...       ...   
300148  300148   Vistara   UK-822     Chennai        Hyderabad     10.08   
300149  300149   Vistara   UK-826     Chennai        Hyderabad     10.42   
300150  300150   Vistara   UK-832     Chennai        Hyderabad     13.83   
300151  300151   Vistara   UK-828     Chennai        Hyderabad     10.00   
300152  300152   Vistara   UK-822     Chennai        Hyderabad     10.08   

        days_left    price  price_per_hour  stops_two_or_more  ...  \
0             1.0

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor

x = data[["stops", "duration", "days_left"]]
y = data["price"]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
)
dummy_model = DummyRegressor(strategy="mean")

dummy_model.fit(x_train, y_train)

dummy_pred = dummy_model.predict(x_test)
dummy_mae = mean_absolute_error(y_test, dummy_pred)

print(dummy_mae)

model = LinearRegression()

model.fit(x_train, y_train)

y_pred = model.predict(x_test)

mae = mean_absolute_error(y_test, y_pred)
print(mae)

ridge_model = Ridge(alpha=1)

ridge_model.fit(x_train,y_train)

y_pred_ridge = ridge_model.predict(x_test)

mae_ridge = mean_absolute_error(y_test, y_pred_ridge)

print(mae_ridge)

model_rand_forest = RandomForestRegressor(n_estimators=100, random_state=42)
model_rand_forest.fit(x_train, y_train)


y_pred_rf = model_rand_forest.predict(x_test)
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print(mae_rf)

19768.69424587037
19133.51120382269
19133.510686581096
